# Effect of varying the magnetic threshold on the CLV profile
Both the thresholding and machine learning pipelines remove limb darkening by dividing the disk by a fitted polynomial profile.
The polynomial is fitted only to quiet-Sun pixels, excluding any with |B| above a threshold so that active regions don't bias the fit.
This notebook tests how the choice of threshold (25, 50, 75 and 100 G) affects the fitted profile.

## 1. Imports:

In [ ]:
import os
import time 

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import cv2
from scipy import ndimage
from scipy.ndimage import label

import sunpy.map
from sunpy.net import Fido, attrs as a
from astropy import units as u
from astropy.time import Time
from reproject import reproject_interp

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

## 2. Configuration:
Edit this cell as needed.

In [ ]:
# Data details
JSOC_EMAIL = "your_email@example.com"
DATA_DIR = './data'
TIME_START = '2014-01-01T00:00:00'
TIME_RANGE = a.Time(TIME_START, Time(TIME_START) + 1 * u.min)

# Required for downloading
MIN_FILE_SIZE = 1_000_000  # In bytes
MAX_RETRIES = 3            

# CLV profile parameters
B_THRESHOLDS = [25, 50, 75, 100] # In Gauss
POLY_ORDER = 5
MU_MIN = 0.1    

# Mu profile plotting
MU_BINS = np.linspace(0, 1, 76)
MU_GRID = np.linspace(0, 1, 200)

os.makedirs(DATA_DIR, exist_ok = True)

## 3. Functions:

In [ ]:
# Function to download FITs files.
def search_and_fetch(query, name, max_retries=MAX_RETRIES):
    for attempt in range(1, max_retries + 1):
        print(f"\n{name}: attempt {attempt}/{max_retries}...")
        result = Fido.search(query)
        n_results = len(result[0]) if len(result) else 0
        print(f"{name}: {n_results} results found")

        if n_results == 0:
            print(f"{name}: no results, retrying search...")
            time.sleep(5)
            continue

        files = Fido.fetch(result, path=f'{DATA_DIR}/{{file}}', overwrite=True)

        good_files = []
        for f in files:
            size = os.path.getsize(f)
            if size > MIN_FILE_SIZE:
                good_files.append(f)
            else:
                print(f"  BAD (size={size} bytes): {f} -- removing")
                os.remove(f)

        if good_files:
            print(f"{name}: SUCCESS -- {len(good_files)} valid file(s)")
            return good_files

        print(f"{name}: no valid files, retrying...")
        time.sleep(5)

    print(f"{name}: FAILED after {max_retries} attempts")
    return []

# Function to remove limb darkening.
def clv_correct(target_map, target_data, mag_map, b_threshold,
                mag_on_target=None, poly_order=POLY_ORDER, mu_min=MU_MIN):
    ny, nx = target_data.shape
    y, x = np.mgrid[0:ny, 0:nx]
    x0 = target_map.meta['crpix1'] - 1
    y0 = target_map.meta['crpix2'] - 1
    r_sun_px = target_map.meta['rsun_obs'] / target_map.meta['cdelt1']
    r = np.sqrt((x - x0)**2 + (y - y0)**2)
    mu = np.sqrt(np.clip(1 - (r / r_sun_px)**2, 0, 1))
    on_disk = r <= r_sun_px
    fit_region = on_disk & (mu >= mu_min)

    if mag_on_target is None:
        mag_on_target, _ = reproject_interp(mag_map, target_map.wcs,
                                            shape_out=target_data.shape)

    quiet_sun_mask = (np.abs(mag_on_target) < b_threshold) & fit_region & np.isfinite(target_data)
    coeffs = np.polyfit(mu[quiet_sun_mask], target_data[quiet_sun_mask], poly_order)
    ctl_profile = np.poly1d(coeffs)(mu)
    ctl_profile[~fit_region] = np.nan
    corrected = target_data / ctl_profile

    return {
        'corrected': corrected,
        'ctl_profile': ctl_profile,
        'quiet_sun_mask': quiet_sun_mask,
        'coeffs': coeffs,
        'mu': mu,
        'on_disk': on_disk,
        'fit_region': fit_region,
    }

# Function build the mu profile.
def build_mu_profile(data, mu, on_disk, mu_bins):
    mu_centres, profile = [], []
    valid = on_disk & np.isfinite(data)
    for i in range(len(mu_bins) - 1):
        lo, hi = mu_bins[i], mu_bins[i + 1]
        mask = (mu >= lo) & (mu < hi) & valid
        if np.sum(mask) > 0:
            profile.append(np.nanmean(data[mask]))
            mu_centres.append((lo + hi) / 2)
    return np.array(mu_centres), np.array(profile)

# Function for plotting the profiles.
def plot_clv_single(res, data, name, b):
    poly = np.poly1d(res['coeffs'])
    norm = poly(1.0)
    centres, profile = build_mu_profile(data, res['mu'],
                                        res['on_disk'] & res['quiet_sun_mask'], MU_BINS)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=centres, y=profile / norm, mode='markers',
                             name='Quiet-Sun mean', marker=dict(size=6)))
    fig.add_trace(go.Scatter(x=MU_GRID, y=poly(MU_GRID) / norm, mode='lines',
                             name=f'Degree-{POLY_ORDER} fit',
                             line=dict(color='red', width=2)))
    fig.update_layout(title=f'{name} CLV, |B| < {b} G',
                      xaxis_title='μ = cos(θ)', yaxis_title='Normalised intensity',
                      width=1000, height=600, template='plotly_white')
    fig.update_xaxes(range=[0, 1.05])
    fig.show()

# Function to set uo the various plots in tabs.
def clv_tabs(results, data, name):
    tab = widgets.Tab()
    outputs = []
    for b, res in results.items():
        out = widgets.Output()
        with out:
            plot_clv_single(res, data, name, b)
        outputs.append(out)
    tab.children = outputs
    for i, b in enumerate(results):
        tab.set_title(i, f'|B| < {b} G')
    return tab

## 4. Downloading FITs files:

In [ ]:
query_aia = TIME_RANGE & a.jsoc.Series('aia.lev1_uv_24s') & a.jsoc.Wavelength(1700*u.angstrom) & a.jsoc.Notify(JSOC_EMAIL)
query_ic  = TIME_RANGE & a.jsoc.Series('hmi.Ic_720s') & a.jsoc.Notify(JSOC_EMAIL)
query_mag = TIME_RANGE & a.jsoc.Series('hmi.M_720s') & a.jsoc.Notify(JSOC_EMAIL)

files_aia = search_and_fetch(query_aia, "AIA 1700")
files_ic  = search_and_fetch(query_ic, "HMI Ic")
files_mag = search_and_fetch(query_mag, "HMI Mag")

if not (files_aia and files_ic and files_mag):
    raise RuntimeError("One or more downloads failed -- check summary above before continuing")

## 5. Loading raw disk images as sunpy maps:

In [ ]:
# Make the sunpy maps
aia_map = sunpy.map.Map(files_aia[0])
ic_map  = sunpy.map.Map(files_ic[0])
mag_map = sunpy.map.Map(files_mag[0])

# Get the data
aia_data = aia_map.data.astype(float)
ic_data  = ic_map.data.astype(float)
mag_data = mag_map.data.astype(float)

## 6. Limb darkening removal:

In [ ]:
# Call functions to reproject and apply CLV correction
mag_on_aia, _ = reproject_interp(mag_map, aia_map.wcs, shape_out=aia_data.shape)
ic_results  = {b: clv_correct(ic_map, ic_data, mag_map, b, mag_on_target=mag_data) for b in B_THRESHOLDS}
aia_results = {b: clv_correct(aia_map, aia_data, mag_map, b, mag_on_target=mag_on_aia) for b in B_THRESHOLDS}

## 7. Plotting:

In [ ]:
plot_tabs = widgets.Tab(children=[clv_tabs(ic_results, ic_data, "HMI Ic"), clv_tabs(aia_results, aia_data, "AIA 1700")])
plot_tabs.set_title(0, 'HMI Ic')
plot_tabs.set_title(1, 'AIA 1700')
display(plot_tabs)